> **中文阅读版**：本文件是 `01_library_overview.ipynb` 的 Markdown 翻译副本。代码、输出、图表和路径保持原样；专有名词和标识符保留英文。
# 因果推断库选择指南

**第 15 章：使用 ML 进行因果估计**
**Docker 镜像**：`ml4t`
**章节参考**：方法选择指南见第 15.1 节（表 15.1）

## 目的
第 15 章使用五个 Python 库，它们彼此并不竞争。
本 notebook 将因果问题映射到能回答该问题的库，并展示每个库期望的 API：它分别将什么作为处理、结果、控制变量和图。

## 为什么下面的估计不是基准

同一个合成样本会跑过多个库，这容易让人做排名。
有四类差异使这种排名无法成立：

1. **问题不同。** 效应估计（EconML、DoWhy、CausalML）从因果结构出发并量化效应。结构发现（Tigramite、causal-learn）从数据出发并提出结构。
2. **处理类型不同。** EconML 和 DoWhy 可直接处理连续处理变量；CausalML 是为二元和离散干预构建的。
3. **估计目标不同。** **估计目标（estimand）** 是分析所瞄准的量。为了适配 CausalML 而将连续处理二值化，会把估计目标从边际效应变为高组与低组之间的对比。
4. **假设不同。** 每个库都要求以不同方式说明数据如何生成以及仍存在何种混杂。

## 学习目标
完成本 notebook 后，你将能够：
- LO1：将因果问题匹配到能回答它的库
- LO2：将表 15.1 中的选择指南应用到自己的问题
- LO3：读懂每个库期望的 API，以及哪个参数承载控制变量
- LO4：区分效应估计与结构发现

## 交叉引用
- **上游**：无（具有已知数据生成过程的合成数据）
- **下游**：每个库都有自己的 notebook
  - EconML：[`03_econml_dml`](03_econml_dml.ipynb)、[`04_dml_crypto_regime`](04_dml_crypto_regime.ipynb)
  - DoWhy：[`02_dowhy_causal_graph`](02_dowhy_causal_graph.ipynb)
  - BSTS：[`06_fed_announcement_bsts`](06_fed_announcement_bsts.ipynb)
  - Tigramite：[`07_tigramite_time_series`](07_tigramite_time_series.ipynb)
  - causal-learn：[`08_neural_causal_discovery`](08_neural_causal_discovery.ipynb)

## 涵盖的库
**效应估计**，在给定因果结构时：
1. **EconML**（Microsoft）——用于连续或二元处理的双重机器学习（DML）和元学习器（metalearners）
2. **DoWhy**（Microsoft/Amazon）——显式图、识别、反驳检验

**结构发现**，用于提出因果图：
3. **Tigramite**——用于时间序列的 PCMCI
4. **causal-learn**（CMU）——PC、FCI、GES、LiNGAM 和 Granger，用于同期与滞后结构

**另作参考**：CausalML（二元处理，本章未使用）、tfp-causalimpact（用于事件研究的贝叶斯结构时间序列，在 `06_fed_announcement_bsts` 中使用）

**先决条件**：无

## 因果设计契约

| 元素 | 本 notebook |
|------|-------------|
| 单元 | 合成观测（从已知 DGP 生成的 1,000 行 i.i.d. 数据中的 1 行） |
| 处理 | `momentum`（连续），部分由 `volatility` 和 `regime` 驱动 |
| 结果 | `returns`（连续），由处理与混杂因素驱动；真实平均处理效应（ATE）在下文绑定到 `TRUE_ATE` |
| 控制变量（EconML 中的 W） | `volatility`、`regime`——二者都会混杂处理—结果路径 |
| 效应修饰变量（X） | 无；这里的目标是常数 ATE，而 `04_dml_crypto_regime` 使用 X 刻画 regime 异质性 |
| 识别假设 | 合成 DGP 完全可观测，因此按设计不存在未观测混杂 |
| 主要失效模式 | 识别层面无；这是 API 冒烟测试，而非经验发现 |

## 设置

In [1]:
"""Causal Inference Library Decision Guide - choose the right causal library for your problem."""

import warnings
from importlib.metadata import PackageNotFoundError, version

import networkx as nx
import numpy as np
import pandas as pd

from utils.reproducibility import set_global_seeds

# Two third-party import-time warnings, each silenced by category and module: DoWhy 0.14
# compiles regexes and docstrings with unescaped backslashes, and pydot calls pyparsing
# methods pyparsing has renamed. Convergence and numerical warnings stay visible.
warnings.filterwarnings("ignore", category=SyntaxWarning, module=".*dowhy")
warnings.filterwarnings("ignore", category=DeprecationWarning, module="pydot")

# networkx 3.x removed nx.algorithms.d_separated; DoWhy 0.14 still calls it.
# Bind the renamed replacement to the old name before importing DoWhy.
if not hasattr(nx.algorithms, "d_separated"):
    nx.algorithms.d_separated = nx.d_separation.is_d_separator

In [2]:
# Production defaults - Papermill injects overrides for CI
SEED = 42

In [3]:
set_global_seeds(SEED)
rng = np.random.default_rng(SEED)

## 检查库可用性

一个库只要能导入就算可用。版本来自已安装发行版的元数据，而不是包本身，因为这五个包中有三个不暴露版本属性。

In [4]:
lib_checks = [
    ("EconML", "econml", "econml"),
    ("DoWhy", "dowhy", "dowhy"),
    ("CausalML", "causalml", "causalml"),
    ("Tigramite", "tigramite", "tigramite"),
    ("causal-learn", "causallearn", "causal-learn"),
]

libraries = {}
for name, module_name, dist_name in lib_checks:
    try:
        __import__(module_name)
    except ImportError:
        libraries[name] = None
        continue
    try:
        libraries[name] = version(dist_name)
    except PackageNotFoundError:
        libraries[name] = "installed"

status_df = pd.DataFrame(
    [
        {"Library": k, "Version": v or "not installed", "Available": v is not None}
        for k, v in libraries.items()
    ]
).set_index("Library")
status_df

,Version,Available
Library,,
EconML,0.17.0,True
DoWhy,0.14,True
CausalML,0.17.0,True
Tigramite,5.2.10.1,True
causal-learn,0.1.4.7,True


## 生成通用演示数据

一个合成数据集会跑过每个库，以便 API 调用并排展示。
`volatility` 和 `regime` 同时驱动处理与结果，这使它们成为混杂因素：对 `returns` 关于 `momentum` 的未调整回归会同时捕捉到它们的影响以及处理的影响。

In [5]:
n = 1000

# Confounders
volatility = rng.exponential(0.02, n)
regime = rng.binomial(1, 0.6, n)

# Treatment (momentum) - depends on the confounders
momentum = 0.5 * regime - 2 * volatility + rng.normal(0, 0.1, n)

# Outcome (returns) - depends on the confounders and on the treatment
TRUE_ATE = 0.02  # the coefficient on momentum, which every estimator below targets
returns = TRUE_ATE * momentum + 0.1 * regime - 3 * volatility + rng.normal(0, 0.02, n)

df = pd.DataFrame(
    {
        "momentum": momentum,
        "returns": returns,
        "volatility": volatility,
        "regime": regime,
    }
)

print(f"Test data: {df.shape[0]:,} obs, true ATE = {TRUE_ATE}")
df.head()

Test data: 1,000 obs, true ATE = 0.02


,momentum,returns,volatility,regime
0,0.458364,-0.002537,0.048084,1
1,0.429718,-0.032758,0.046724,1
2,0.501127,0.014741,0.047695,1
3,0.497595,0.083456,0.005596,1
4,0.535744,0.090199,0.001729,1


处理所携带的独立变异非常少：一旦将 `volatility` 和 `regime` 的影响剔除（partialled out），`momentum` 剩下的就是其自身的噪声项。决定这些估计器中任何一个能以多高精度测量 ATE 的，是这一点，而不是效应大小，因此下文每个估计都报告一个区间。

## 1. EconML（Microsoft）

EconML 拟合**双重机器学习**（double machine learning，DML）：一个模型根据控制变量预测结果，另一个模型根据控制变量预测处理变量，而处理效应则由两者各自的残差估计得到。它接受连续型或二元处理变量，并可以建模异质性效应，因此当问题在于效应有多大以及效应如何变化时，它是合适的工具。

当调整集（adjustment set）已知但因果图尚未写出来时，使用它。

In [6]:
if libraries["EconML"]:
    from econml.dml import LinearDML
    from sklearn.ensemble import GradientBoostingRegressor

    Y = df["returns"].to_numpy()
    T = df["momentum"].to_numpy()
    W = df[["volatility", "regime"]].to_numpy()  # controls used for residualization

    dml = LinearDML(
        model_y=GradientBoostingRegressor(n_estimators=50, max_depth=3, random_state=SEED),
        model_t=GradientBoostingRegressor(n_estimators=50, max_depth=3, random_state=SEED),
        cv=3,
        random_state=SEED,
    )
    dml.fit(Y, T, W=W)

    econml_ate = float(dml.ate())
    econml_lo, econml_hi = (float(b) for b in dml.ate_interval(alpha=0.05))
    print(f"EconML ATE: {econml_ate:.6f}  95% CI [{econml_lo:.6f}, {econml_hi:.6f}]")
else:
    econml_ate = econml_lo = econml_hi = None
    print("EconML not available")

EconML ATE: 0.028640  95% CI [0.016486, 0.040794]


混淆变量在 EconML 的 DML 接口中以 `W` 的形式进入，即用于残差化的控制变量，而不是以 `X` 的形式进入，后者是用于建模处理效应异质性的效应修饰变量。对于单个调整后的 ATE，这一区分不会改变答案；但一旦问题变成效应是否在不同 regime 下存在差异，它就会产生影响；`04_dml_crypto_regime` 使用了 `X` 角色。

`cv=3` 会随机划分样本，这在这里是有效的，因为按构造这些行是独立同分布（i.i.d.）的。时间序列需要尊重顺序的划分，否则结果模型会从未来学习；`03_econml_dml` 则改为采用前向游走（walk forward）方式。

## 2. DoWhy（Microsoft/Amazon）

DoWhy 会先要求提供因果图，从图中推导出估计目标（estimand），然后才进行估计。把图写下来正是关键所在：它使调整集成为所述假设的结果，而不是一种选择，并为反驳测试（refutation tests）提供可扰动的对象。

当假设必须可见且可检验时使用它，在研究环境中这大多数时候都成立。

In [7]:
if libraries["DoWhy"]:
    from dowhy import CausalModel

    # Specify the causal graph
    causal_graph = """
    digraph {
        volatility -> momentum;
        volatility -> returns;
        regime -> momentum;
        regime -> returns;
        momentum -> returns;
    }
    """

    model = CausalModel(
        data=df,
        treatment="momentum",
        outcome="returns",
        graph=causal_graph,
    )

    identified_estimand = model.identify_effect()
    estimate = model.estimate_effect(
        identified_estimand,
        method_name="backdoor.linear_regression",
    )

    dowhy_ate = float(estimate.value)
    dowhy_lo, dowhy_hi = (float(b) for b in np.asarray(estimate.get_confidence_intervals()).ravel())
    print(f"DoWhy ATE: {dowhy_ate:.6f}  95% CI [{dowhy_lo:.6f}, {dowhy_hi:.6f}]")
else:
    dowhy_ate = dowhy_lo = dowhy_hi = None
    print("DoWhy not available")

DoWhy ATE: 0.028835  95% CI [0.016772, 0.040899]


### 估计目标 vs. 估计量

**估计目标**（estimand）是所要瞄准的量；**估计量**（estimator）是为其产生数值的过程。两个库可能报告不同的数值，原因可能是它们对同一估计目标使用了不同的估计量，也可能是因为它们瞄准的根本就是不同的估计目标；只有前一种情况才是关于准确性的问题。

| 估计目标 | 描述 | 库 |
|---|---|---|
| $E[Y \mid do(T = t+1)] - E[Y \mid do(T = t)]$ | 连续处理变量的边际效应 | EconML, DoWhy |
| $E[Y \mid do(T = 1)] - E[Y \mid do(T = 0)]$ | 二元处理效应，高组对低组 | CausalML |

如果估计目标无法从手头数据中识别，就没有任何估计量能恢复它。将灵活模型拟合到有混淆的比较上，只会对错误的问题给出精确的答案。

## 3. 本章引用的其他库

**CausalML**（Uber）面向 uplift，使用 S-、T-、X- 和 R-learner，并且对实验和营销中典型的二元或离散干预支持最强。连续处理变量虽然可行，但并非该库的主攻方向。第 15 章涉及连续处理变量和事件研究，因此 CausalML 出现在选择指南中，但没有完整示例；[CausalML 文档](https://causalml.readthedocs.io/)涵盖了 uplift 场景。

**tfp-causalimpact** 拟合贝叶斯结构时间序列模型，并需要一个结果序列的预期间来构建反事实，而这里的横截面样本无法提供该预期间。`06_fed_announcement_bsts` 在美联储公告上运行它。

## 4. 库比较总结

表中列出了本章引用的每个库以及各自的使用位置。

In [8]:
def build_library_comparison():
    """Build library comparison table showing Chapter 15 coverage."""
    comparison_data = [
        {
            "Library": "EconML",
            "Use": "Effect estimation",
            "Treatment": "Continuous/Binary",
            "Key Feature": "DML, metalearners",
            "Notebook": "03_econml_dml, 04_dml_crypto_regime",
        },
        {
            "Library": "DoWhy",
            "Use": "Effect estimation",
            "Treatment": "Continuous/Binary",
            "Key Feature": "Graph, identification, refutation",
            "Notebook": "02_dowhy_causal_graph",
        },
        {
            "Library": "CausalML",
            "Use": "Uplift/CATE",
            "Treatment": "Binary/discrete (focus)",
            "Key Feature": "S/T/X/R learners",
            "Notebook": "(not used in this chapter)",
        },
        {
            "Library": "tfp-causalimpact",
            "Use": "Event study",
            "Treatment": "Binary event",
            "Key Feature": "BSTS counterfactual",
            "Notebook": "06_fed_announcement_bsts",
        },
        {
            "Library": "Tigramite",
            "Use": "Time-series discovery",
            "Treatment": "N/A",
            "Key Feature": "PCMCI",
            "Notebook": "07_tigramite_time_series",
        },
        {
            "Library": "causal-learn",
            "Use": "Discovery",
            "Treatment": "N/A",
            "Key Feature": "PC, FCI, GES, LiNGAM, Granger",
            "Notebook": "08_neural_causal_discovery",
        },
    ]
    return pd.DataFrame(comparison_data).set_index("Library")


comparison_df = build_library_comparison()
comparison_df

,Use,Treatment,Key Feature,Notebook
Library,,,,
EconML,Effect estimation,Continuous/Binary,"DML, metalearners","03_econml_dml, 04_dml_crypto_regime"
DoWhy,Effect estimation,Continuous/Binary,"Graph, identification, refutation",02_dowhy_causal_graph
CausalML,Uplift/CATE,Binary/discrete (focus),S/T/X/R learners,(not used in this chapter)
tfp-causalimpact,Event study,Binary event,BSTS counterfactual,06_fed_announcement_bsts
Tigramite,Time-series discovery,N/A,PCMCI,07_tigramite_time_series
causal-learn,Discovery,N/A,"PC, FCI, GES, LiNGAM, Granger",08_neural_causal_discovery


Tigramite 和 causal-learn 学习图；EconML 和 DoWhy 则在图或调整集确定后量化效应。因此，发现先于估计，而不是取代估计；其输出是一组待检验的假设，而不是一个可信赖的结构。

本章的发现方法中有两个不是库调用。NOTEARS 通过可微无环性约束下的连续优化来学习同期图，它按照 Zheng 等（2018）直接实现在 `08_neural_causal_discovery` 中；causal-learn 并未内置它。Granger 因果性在那里作为预测基线。causal-learn 提供 VAR-LiNGAM 拟合，它从非高斯残差中恢复滞后和同期结构。

## 5. 选择指南

第 15.1 节将选择整理为表格，此处复现。首要问题是因果图是否已知，因为这决定了问题是估计还是发现。

| 因果图已知？ | 处理类型 | 方法 | 库 |
|---|---|---|---|
| 是 | 二元或连续 | 后门调整 | DoWhy |
| 是 | 连续 | DML | EconML |
| 是 | 二元，在某一时点 | BSTS | tfp-causalimpact |
| 是 | 二元或离散，uplift | 元学习器（Metalearners） | CausalML |
| 否 | 不适用 | PCMCI | Tigramite |
| 否 | 不适用 | VAR-LiNGAM | causal-learn |
| 否 | 不适用 | NOTEARS | 在 `08_neural_causal_discovery` 中实现 |

解读上半部分：DoWhy 和 EconML 都需要一个能够识别效应的调整集，区别在于调整集的提供方式。DoWhy 接收因果图并从中推导调整集；EconML 直接接收控制变量，并用机器学习模型拟合它们。两者都无法绕开识别假设，因此“无因果图”是先运行发现方法的理由，而不是在两种估计器之间偏好其一的理由。

解读下半部分：划分依据是同期结构与滞后结构，而不是横截面与时间序列。PCMCI 和 VAR-LiNGAM 都读取时间滞后依赖，`08_neural_causal_discovery` 在同一投资范围上运行它们；NOTEARS 则针对单个横截面内的结构。

## 6. 效应估计及其可支持的结论

EconML 和 DoWhy 在此针对同一估计目标（estimand），因此二者的估计值可以相互比较，也可以与已知的真实 ATE 比较。每个估计值都报告置信区间，最后一列记录该区间是否覆盖真实值。

In [9]:
rows = [
    {
        "Method": "True ATE",
        "Estimate": TRUE_ATE,
        "95% CI": "-",
        "Abs. error": 0.0,
        "Covers true ATE": "-",
    }
]
if econml_ate is not None:
    rows.append(
        {
            "Method": "EconML (DML)",
            "Estimate": econml_ate,
            "95% CI": f"[{econml_lo:.4f}, {econml_hi:.4f}]",
            "Abs. error": abs(econml_ate - TRUE_ATE),
            "Covers true ATE": bool(econml_lo <= TRUE_ATE <= econml_hi),
        }
    )
if dowhy_ate is not None:
    rows.append(
        {
            "Method": "DoWhy (Backdoor)",
            "Estimate": dowhy_ate,
            "95% CI": f"[{dowhy_lo:.4f}, {dowhy_hi:.4f}]",
            "Abs. error": abs(dowhy_ate - TRUE_ATE),
            "Covers true ATE": bool(dowhy_lo <= TRUE_ATE <= dowhy_hi),
        }
    )

estimates_df = pd.DataFrame(rows).set_index("Method")
estimates_df

,Estimate,95% CI,Abs. error,Covers true ATE
Method,,,,
True ATE,0.020000,-,0.000000,-
EconML (DML),0.028640,"[0.0165, 0.0408]",0.008640,True
DoWhy (Backdoor),0.028835,"[0.0168, 0.0409]",0.008835,True


先看覆盖列，再看误差列。绝对误差混合了抽样噪声与偏差，单一估计无法将二者分开，因此估计值与真实 ATE 之间的差距本身并不能证明调整失败。区间体现的是估计器对自身精度的说明：只要估计值的区间覆盖真实值，就与该真实值一致，无论点估计离得多远。未覆盖并不是上述情形的镜像。一个按 20 个样本中覆盖 19 个来构造的区间，即使调整完全正确，也会在第 20 个样本中未覆盖，因此单次未覆盖本身不能说明任何问题。这些区间究竟是否经过校准，是关于重复抽样的问题，而本 notebook 只抽取了一个样本。

单样本上的覆盖情况也是这两行最多所能支持的结论。它们拟合于同一批 1,000 个观测，因此误差会同向变动，单次抽样无法对二者进行排序。CausalML 在二值化处理上回答的是另一个问题，因此其数值不可比较，表中未将其纳入。

## 关键要点

### 选择库

1. **问题决定方法族。** 给定结构下的效应估计：EconML、DoWhy、CausalML。结构发现：Tigramite、causal-learn。
2. **处理类型决定方法族内的工具。** 连续处理：EconML DML 或 DoWhy。二元或离散 uplift：CausalML 或 EconML。单一有日期的事件：BSTS。
3. **数据形态决定发现方法。** 多变量序列中的滞后依赖：PCMCI 或 VAR-LiNGAM。横截面内的同期结构：NOTEARS。

### 方法学说明

- **不同估计目标不可比较。** 连续处理得到的 ATE 与二值化处理得到的 uplift 估计回答的是不同问题。
- **发现先于估计。** 发现的图是一个假设；后续估计器会继承发现步骤中出错的任何部分。
- **二者可结合。** 在 DoWhy 中指定图并用 EconML 进行估计是一种常见模式，且 DoWhy 可以直接调用 EconML 估计器。

### 深入阅读

本 notebook 展示 API 形态。后续 notebook 会补充其略去的部分：时间切分、考虑面板结构的标准误、反驳检验与敏感性分析。
- **DML**：[`03_econml_dml`](03_econml_dml.ipynb)、[`04_dml_crypto_regime`](04_dml_crypto_regime.ipynb)
- **BSTS 事件研究**：[`06_fed_announcement_bsts`](06_fed_announcement_bsts.ipynb)
- **DoWhy 反驳检验**：[`02_dowhy_causal_graph`](02_dowhy_causal_graph.ipynb)
- **时间序列发现**：[`07_tigramite_time_series`](07_tigramite_time_series.ipynb)
- **NOTEARS 与 VAR-LiNGAM**：[`08_neural_causal_discovery`](08_neural_causal_discovery.ipynb)